# IPAVentures horizon scan — run `2026-09-06T2100`

Generated 2026-09-06 22:38 UTC by `python -m src.notebook --run-id 2026-09-06T2100`.

This notebook is the audit trail for one horizon-scanning run. It reads a
frozen corpus out of DuckDB and walks the pipeline stage by stage: what went
in, what each stage did to it, and what came out. Every code cell is runnable
against the same database, and the outputs shown were produced by running them.

**What this notebook establishes, and what it does not.**

It establishes that the published shortlist follows *deterministically* from
this corpus and these weights. Four numbers — the emergence score, the Three
Horizons band, the opportunity index and the composite rank — are recomputed
from their stored inputs and checked against what the pipeline stored, so the
arithmetic is not something you have to take on trust.

It establishes nothing whatsoever about whether the weights are *right*. No
weight in this pipeline has been validated against a known past opportunity.
The ranking is a hypothesis about ranking, and the useful argument to have over
this document is about the corpus and the weights, not about the arithmetic.

Three caveats that survive every number below:

1. **The opportunity index is not a market size.** It is a relative, unitless,
   within-run ordering and cannot be converted into a dollar figure.
2. **Scores are relative to this run's population.** Emergence and composite
   scores are percentile-ranked within the run; they are not comparable to
   another run's unless the corpus and config snapshot match.
3. **The scan cannot find what the scan frame does not ask for.** A topic's
   absence from this document is not evidence of its absence from the world.

**Reproducing this.** Open the notebook from inside a checkout of the
repository with `data/bigthink.duckdb` present, and run all cells. Stage 1 is
deliberately not re-executed — collectors hit live rate-limited APIs, so
re-running collection would produce a different corpus and quietly invalidate
every comparison below.

## Setup

Open the corpus read-only and locate the repository.

In [1]:
import json
import sys
from pathlib import Path

import duckdb
import numpy as np

# Find the repository, so this notebook runs from wherever it has been opened.
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "bigthink_config.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "bigthink_config.yaml").exists():
    raise SystemExit(
        "Could not locate the BigThink repository from the current directory. "
        "Open this notebook from inside a checkout of the repo."
    )
sys.path.insert(0, str(REPO_ROOT))

from src import db
from src.notebook import diff_config, table, verify_close, verify_identical

RUN_ID = '2026-09-06T2100'
DB_PATH = REPO_ROOT / 'data/bigthink.duckdb'

# Read-only, deliberately. This notebook exists to explain a run; it must not
# be able to alter the corpus that run was computed from.
conn = duckdb.connect(str(DB_PATH), read_only=True)

print(f"run_id    {RUN_ID}")
print(f"database  {DB_PATH}")
print(f"documents {conn.execute('SELECT count(*) FROM documents').fetchone()[0]:,}")

run_id    2026-09-06T2100
database  /home/runner/work/BigThink/BigThink/data/bigthink.duckdb
documents 17,762


## Provenance — what produced these numbers

Every stage writes a row to `pipeline_runs` carrying a full snapshot of the
config it ran under. That snapshot, not the file currently on disk, is what
this notebook verifies against: a threshold edited after the run must not be
able to change what "reproduced" means retrospectively.

In [2]:
# The weights checked throughout this notebook come from the run itself, not
# from today's bigthink_config.yaml. Editing a threshold after a run must not
# quietly change what "reproduced" means.
snapshot = conn.execute(
    """
    SELECT stage, config_snapshot FROM pipeline_runs
    WHERE run_id = ? AND config_snapshot IS NOT NULL AND config_snapshot <> ''
    ORDER BY id DESC LIMIT 1
    """,
    [RUN_ID],
).fetchone()
if not snapshot:
    raise SystemExit(f"No config snapshot recorded for run_id={RUN_ID!r}.")

CONFIG = json.loads(snapshot[1])
ROTOLO_WEIGHTS = CONFIG["emergence"]["rotolo_weights"]
RANK_WEIGHTS = CONFIG["synthesis"]["rank_weights"]

print(f"Config recovered from the {snapshot[0]!r} stage log.\n")
print("Embedding backend :", CONFIG["embeddings"]["backend"])
_topics_cfg = CONFIG["emergence"]["topics"]
_backend = CONFIG["embeddings"]["backend"]
_method = _topics_cfg.get("method", "agglomerative")
# Runs before 2026-08-30 stored one threshold map keyed by backend alone, when
# there was only one numpy method. Read either shape, so an old notebook still
# reports the number its own run actually used.
_thresh = (_topics_cfg.get("similarity_thresholds", {}).get(_method)
           or _topics_cfg.get("similarity_threshold_by_backend", {}))
print("Clustering method :", _method)
if _method == "bertopic":
    # Under BERTopic the similarity threshold does not cluster anything —
    # HDBSCAN works on density in UMAP space and takes no cosine cut-off. It is
    # read for one purpose only: attaching documents from non-forming sources
    # to the nearest finished topic. Labelled for what it does, so a reviewer
    # is not left comparing it against a clustering threshold from another run.
    print("Attachment thresh :", _thresh.get(_backend, "not recorded"),
          f"x ratio {_topics_cfg.get('attachment_threshold_ratio', 0.6)}")
    # THE SEED. UMAP's initialisation is stochastic, so this is the difference
    # between a topic set another analyst can reproduce and one they cannot.
    # Printed from the run's own snapshot, never from today's config file.
    _bt = _topics_cfg.get("bertopic") or {}
    print("BERTopic seed     :", _bt.get("random_state", "NOT RECORDED"))
    print("BERTopic UMAP     :",
          f"n_neighbors={_bt.get('n_neighbors')}, "
          f"n_components={_bt.get('n_components')}, "
          f"min_dist={_bt.get('min_dist')}, metric={_bt.get('metric')}")
    print("BERTopic HDBSCAN  :",
          f"min_cluster_size={_bt.get('min_cluster_size') or _topics_cfg.get('min_topic_size')}, "
          f"min_samples={_bt.get('min_samples')}, "
          f"selection={_bt.get('cluster_selection_method')}")
    if not isinstance(_bt.get("random_state"), int):
        print("\n  WARNING: no seed recorded for this run. UMAP is stochastic, "
              "so the topics below cannot be reproduced exactly, even from this "
              "same corpus and config.")
else:
    print("Clustering thresh :", _thresh.get(_backend, "not recorded"))
print("Time slice        :", CONFIG["emergence"]["time_slice"])
print("Collection window :",
      CONFIG["collection"]["start_year"], "-", CONFIG["collection"]["end_year"])
print()
print("Rotolo weights (emergence):",
      ", ".join(f"{k} {v}" for k, v in sorted(ROTOLO_WEIGHTS.items())))
print("Rank weights   (shortlist):",
      ", ".join(f"{k} {v}" for k, v in sorted(RANK_WEIGHTS.items())))

Config recovered from the 'stage4_opportunity_index' stage log.

Embedding backend : bge
Clustering method : bertopic
Attachment thresh : 0.9 x ratio 0.6
BERTopic seed     : 42
BERTopic UMAP     : n_neighbors=15, n_components=5, min_dist=0.0, metric=cosine
BERTopic HDBSCAN  : min_cluster_size=8, min_samples=None, selection=eom
Time slice        : year
Collection window : 2018 - 2026

Rotolo weights (emergence): coherence 0.15, growth 0.3, impact 0.2, novelty 0.25, uncertainty 0.1
Rank weights   (shortlist): asset_leverage 0.25, emergence 0.4, strategic_fit 0.35


### Stage execution log

What ran, over how much, and how it ended.

In [3]:
print(table(
    conn.execute(
        """
        SELECT stage, status, records_in, records_out,
               round(date_diff('millisecond', started_at, finished_at) / 1000.0, 1) AS seconds,
               message
        FROM pipeline_runs WHERE run_id = ? ORDER BY id
        """,
        [RUN_ID],
    ).fetchall(),
    ["stage", "status", "in", "out", "seconds", "message"],
    max_width=64,
))

stage                     status      in   out   seconds  message
------------------------  -------  -----  ----  --------  ----------------------------------------------------------------
stage0_strategy           success      —    34     0.400  34 refs (9 objectives, 6 initiatives, 7 critical-tech fields, 1…
stage1_collect            partial  14621  1523  5297.600  14621 fetched, 1523 new, 1 failed/skipped pairs, 17 partial in …
stage2_emergence          success  17762   117   524.400  117 topics from 17762 documents across 9 slices (2018-2026); 29…
stage5_synthesis          success    117   117    46.400  ranked 117 topics; outputs written
stage3_scoring            success    117   117    44.100  scored 117 topics; mean fit 0.492, mean leverage 0.360, 0 criti…
stage4_opportunity_index  success    117   117     0.200  indexed 117 of 117 topics (0 suppressed as too thin)


### Has the repository moved since this run?

If any value below differs, the code you are reading is not quite the code that produced these scores.

In [4]:
from src.config import load_config, snapshot_config

# Has the repository moved since this run? Comparing scores across configs is
# the mistake this method is most exposed to, so the notebook checks rather
# than assuming.
current = json.loads(snapshot_config(load_config()))
differences = diff_config(CONFIG, current)
if not differences:
    print("The config on disk is identical to the one that produced this run.")
else:
    print(f"{len(differences)} config value(s) have changed since this run.")
    print("Scores below were computed under the STORED values, not these.\n")
    print(table(
        [(p, w, n) for p, w, n in differences[:30]],
        ["setting", "at run time", "on disk now"],
        max_width=44,
    ))

The config on disk is identical to the one that produced this run.


## Stage 0 — Strategy encoding

The published strategy is turned into a reference set: corporate-plan
objectives and initiatives, DISR critical-technology fields, and an inventory
of what IP Australia would bring to an opportunity. Each reference carries a
text body (embedded into a vector) and a lexicon of terms.

This is the yardstick every strategic-fit and asset-leverage score is measured
against, which makes it the most consequential reviewable artefact in the repo
after the scan frame. The files are `data/strategy/*.yaml` and they are meant
to be critiqued without reading any Python.

In [5]:
print(table(
    conn.execute(
        """
        SELECT ref_type, count(*) AS refs, round(avg(weight), 2) AS avg_weight
        FROM strategy_refs GROUP BY ref_type ORDER BY ref_type
        """
    ).fetchall(),
    ["reference type", "count", "avg weight"],
))
print()
print("Every strategic-fit score below is a similarity against one of these.")
print("A topic the strategy does not describe cannot score highly on fit, however")
print("important it is — that is a property of the instrument, not a finding.\n")
print(table(
    conn.execute(
        "SELECT ref_type, code, label FROM strategy_refs ORDER BY ref_type, code, ref_id"
    ).fetchall(),
    ["type", "code", "label"],
    max_width=58,
))

reference type  count  avg weight
--------------  -----  ----------
asset              12       0.920
critical_tech       7       1.000
initiative          6       1.000
objective           9       1.020

Every strategic-fit score below is a similarity against one of these.
A topic the strategy does not describe cannot score highly on fit, however
important it is — that is a property of the instrument, not a finding.

type           code             label
-------------  ---------------  ----------------------------------------------------------
asset          AS-CONVENING     Convening power and international IP standing
asset          AS-DATAPIPE      Zero-budget analytical pipeline pattern
asset          AS-EXAMINATION   Examination workforce and AI/ADM trials
asset          AS-FIRSTNATIONS  First Nations Strategy partnerships
asset          AS-IPFR          IP First Response
asset          AS-IPGOD         IPGOD / IPLoRD longitudinal registry data
asset          AS-OCTAVIUS      Oct

## Stage 1 — Signal collection

Seed queries from `data/strategy/scan_frame.yaml` are issued against each
enabled source; results are normalised, STEEPV-tagged, deduplicated on a stable
`native_id`, and written to DuckDB.

**This stage is not reproducible and is not re-run here.** The collectors call
live APIs under rate limits and daily budgets; a source that was healthy during
the run may be retired or metered now. What follows describes the corpus that
exists, from the collection log the run wrote at the time.

Read this section as the answer to "what could this scan possibly have found?",
because everything downstream is conditioned on it.

In [6]:
# NOTE the difference between these two tables. The corpus ACCUMULATES across
# runs — each run adds to it and growth curves lengthen instead of resetting —
# so the first table is the whole corpus the topics were formed from. The
# second is only what THIS run fetched. They are not meant to reconcile.
print("The corpus these topics were formed from, by source:\n")
print(table(
    conn.execute(
        """
        SELECT source, count(*) AS documents, min(year) AS first_year,
               max(year) AS last_year, count(DISTINCT scan_frame_key) AS frames
        FROM documents GROUP BY source ORDER BY documents DESC
        """
    ).fetchall(),
    ["source", "documents", "first year", "last year", "frames"],
))
print()
print("What this run's collection actually did (per source query):\n")
print(table(
    conn.execute(
        """
        SELECT source, status, count(*) AS queries, sum(records) AS records
        FROM collection_log WHERE run_id = ?
        GROUP BY source, status ORDER BY source, status
        """,
        [RUN_ID],
    ).fetchall(),
    ["source", "status", "queries", "records"],
))

The corpus these topics were formed from, by source:

source     documents  first year  last year  frames
---------  ---------  ----------  ---------  ------
gdelt           9200        2024       2026      17
openalex        3821        2018       2026      20
crossref        2378        2018       2026      13
arxiv           2214        2018       2026       9
datagovau        149        2022       2026       3

What this run's collection actually did (per source query):

source     status   queries  records
---------  -------  -------  -------
arxiv      success        9     2270
crossref   success       13     2498
datagovau  success        3      150
gdelt      failed         1        0
gdelt      partial       17     5796
openalex   success       20     3907


In [7]:
print("Documents per year — the history the growth curves are fitted to:\n")
print(table(
    conn.execute(
        "SELECT year, count(*) FROM documents WHERE year IS NOT NULL "
        "GROUP BY year ORDER BY year"
    ).fetchall(),
    ["year", "documents"],
))
print()
print("STEEPV coverage. The scan frame decides this distribution, and it")
print("mirrors where free structured data exists rather than where")
print("opportunities are:\n")
print(table(
    conn.execute(
        "SELECT steepv, count(*) FROM documents GROUP BY steepv ORDER BY 2 DESC"
    ).fetchall(),
    ["STEEPV", "documents"],
))

Documents per year — the history the growth curves are fitted to:

year  documents
----  ---------
2018        930
2019       1013
2020       1173
2021       1061
2022        961
2023        993
2024       1635
2025       5459
2026       4537

STEEPV coverage. The scan frame decides this distribution, and it
mirrors where free structured data exists rather than where
opportunities are:

STEEPV         documents
-------------  ---------
Technological       7791
Political           2920
Economic            2523
Legal               1808
Values              1076
Social               986
Environmental        658


### What each seed query actually returned

`scan_frame.yaml` is the biggest single determinant of the output — the scan cannot find what the frame does not ask for. A frame that returned almost nothing is a blind spot in this run's evidence, and no amount of re-weighting downstream can compensate for it.

In [8]:
# The scan frame is the single biggest determinant of what this scan could
# possibly have found, so it is worth seeing which seed queries actually
# yielded a corpus and which returned almost nothing. A frame near the bottom
# of this table is a blind spot, whatever the shortlist says.
print(table(
    conn.execute(
        """
        SELECT scan_frame_key, any_value(steepv) AS steepv, count(*) AS documents,
               count(DISTINCT source) AS sources
        FROM documents WHERE scan_frame_key IS NOT NULL
        GROUP BY scan_frame_key ORDER BY documents DESC
        """
    ).fetchall(),
    ["scan frame", "STEEPV", "documents", "sources"],
    max_width=40,
))

scan frame                   STEEPV         documents  sources
---------------------------  -------------  ---------  -------
ct_quantum                   Technological       1599        3
ai_governance_assurance      Political           1529        5
digital_identity_provenance  Technological       1375        3
ai_authorship_inventorship   Legal               1132        4
ct_biotech                   Technological       1100        3
indigenous_knowledge         Values              1076        3
ip_finance_valuation         Economic            1058        3
ip_enforcement_counterfeit   Economic            1003        3
trust_institutions           Social               986        3
gov_service_delivery         Political            891        4
ip_admin_automation          Technological        755        4
ct_ai                        Technological        706        3
geographical_indications     Legal                676        3
ct_energy                    Environmental        658  

> **Noted automatically from this run's own numbers.**
>
> - Failed queries by source: gdelt (1). Each failure costs that frame's contribution to the corpus, so the shortlist is drawn from a slightly different scan than the frame file describes.

## Stage 2 — Emergence detection

Documents are embedded, clustered into topics, sliced into a time series, and
scored on the five attributes of emergence from Rotolo, Hicks & Martin (2015):

| Attribute | Indicator |
|---|---|
| novelty | mean embedding distance from the earlier corpus centroid |
| growth | slice-over-slice CAGR blended with Kleinberg burst intensity |
| coherence | intra-topic cosine density |
| impact | citation percentile, computed *within* source |
| uncertainty | normalised entropy over institutions and source types |

Two choices here shape everything after them, and both are deliberate:

- **Impact percentiles are computed within source.** arXiv reports no
  citations. Ranked globally, every preprint would sit at the bottom and the
  fastest-moving evidence in the corpus would be systematically penalised.
- **A logistic curve is fitted directly rather than by linearising the
  logit.** Linearisation reports an early-exponential topic as *saturated*,
  which inverts the Three Horizons band for precisely the technologies a
  horizon scan exists to find.

In [9]:
topics = db.fetch_topics(conn, RUN_ID)   # emergence_score DESC — Stage 5 read them in this order

print(table(
    [
        (t["topic_id"], (t["label"] or "")[:44], t["document_count"],
         t["novelty"], t["growth"], t["coherence"], t["impact"], t["uncertainty"],
         t["emergence_score"], t["horizon"], t["signal_class"])
        for t in topics
    ],
    ["topic", "label", "docs", "nov", "grw", "coh", "imp", "unc",
     "emergence", "H", "signal"],
    places=2,
))

topic                  label                                         docs   nov   grw   coh   imp   unc  emergence  H   signal
---------------------  --------------------------------------------  ----  ----  ----  ----  ----  ----  ---------  --  ------
2026-09-06T2100-T0093  copyright page / copyright notice               23  0.30  0.12  0.93  0.67  0.43       0.82  H2  weak
2026-09-06T2100-T0036  kilometre km / square kilometre / africa / p    57  0.36  0.20  0.82  0.68  0.30       0.81  H2  strong
2026-09-06T2100-T0107  covid-19 / pandemic / trust / public            20  0.24  0.07  0.88  0.59  0.87       0.80  H1  weak
2026-09-06T2100-T0058  cultural heritage / property cultural / inte    40  0.20  0.18  0.86  0.71  0.67       0.80  H1  weak
2026-09-06T2100-T0095  quantum / entanglement / deterministic / tel    29  0.27  0.05  0.87  0.58  0.94       0.77  H1  weak
2026-09-06T2100-T0022  covid-19 / trust / pandemic / health            88  0.20  0.20  0.80  0.60  0.83       0.77  H1 

### Re-deriving the emergence score

The score above is not an opaque model output. It is a weighted sum of five
percentile-ranked attributes, and the cell below recomputes it from the stored
attributes and the stored weights.

In [10]:
from src.normalise import percentile_rank
from src.stage2_emergence import assign_horizon

ATTRS = ("novelty", "growth", "coherence", "impact", "uncertainty")

# emergence_score is a weighted sum of the five attributes AFTER each has been
# percentile-ranked within this run's population. Rank-normalising is what
# makes the configured weights mean what they say: a weighted sum of raw values
# is dominated by whichever attribute happens to have the widest spread.
#
# The consequence, stated plainly: the score is RELATIVE TO THIS RUN. A run of
# uniformly dull topics still produces one scoring near 1.0.
ranked = {a: percentile_rank([float(t[a]) for t in topics]) for a in ATTRS}
recomputed = [
    sum(ranked[a][i] * float(ROTOLO_WEIGHTS[a]) for a in ATTRS)
    for i in range(len(topics))
]
verify_close("emergence_score", [t["emergence_score"] for t in topics], recomputed)

# The Three Horizons band is a pure function of the fitted logistic maturity
# and two cut-points from the config. Nothing about a topic's age enters it.
verify_identical(
    "Three Horizons band",
    [t["horizon"] for t in topics],
    [assign_horizon(float(t["maturity"]), CONFIG) for t in topics],
)

PASS  emergence_score reproduced for 117 topics — largest deviation 0 (tolerance 1e-09)
PASS  Three Horizons band reproduced — 117 of 117 entries identical


> **Noted automatically from this run's own numbers.**
>
> - 13,503 of 17,762 documents (76%) are attached to any topic. The remainder did not cluster tightly enough to be assigned and are absent from every score below.
>
> - 10 topic(s) sit within 0.05 of a Three Horizons cut-point (artificial intelligence / patent / inven (0.38 → H2); llm / language / task / reasoning (0.71 → H2); adm / ai / decision-making / automated (0.71 → H2); retrieval / rag / text / language (0.71 → H2); copyright page / copyright notice (0.71 → H2); watermark / text / llm / language (0.77 → H1); patent retrieval / retrieval system / in (0.77 → H1); regulatory policy / regulatory reform /  (0.77 → H1); valuation / briefing / valuer / market (0.77 → H1); patent / quantum / ai / invention (0.77 → H1)). Their band is an artefact of where the cut falls, not a finding about them; do not present these as confidently H1/H2/H3.

## Stage 3 — Strategic fit and asset leverage

Two scores per topic, each blending an embedding similarity with a lexical
overlap term so that a topic naming an objective explicitly is not penalised by
embedding drift.

- **Strategic fit** — closeness to the Stage 0 objectives and initiatives.
  Answers "is this on strategy?"
- **Asset leverage** — closeness to IP Australia's own data, capability and
  relationship inventory. Answers "could we credibly act on it?", which is the
  question that separates an interesting trend from a viable venture.

Watch the *spread* of each axis below, not just the values. Both are
rank-normalised before they enter the ranking, so a compressed axis still
contributes its full configured share of the ordering while carrying much less
real information than that share implies.

In [11]:
ranked_topics = db.fetch_ranked_topics(conn, RUN_ID)

print(table(
    [
        (r["topic_id"], (r["label"] or "")[:38], r["strategic_fit"],
         r["best_objective"] or "—", r["asset_leverage"],
         r["critical_tech"] or "—")
        for r in ranked_topics
    ],
    ["topic", "label", "fit", "closest objective", "leverage", "DISR field"],
    max_width=36,
))
print()
fits = [float(r["strategic_fit"] or 0.0) for r in ranked_topics]
levs = [float(r["asset_leverage"] or 0.0) for r in ranked_topics]
print(f"strategic fit  range {min(fits):.3f}–{max(fits):.3f}  median {np.median(fits):.3f}")
print(f"asset leverage range {min(levs):.3f}–{max(levs):.3f}  median {np.median(levs):.3f}")

topic                  label                                   fit  closest objective                     leverage  DISR field
---------------------  ------------------------------------  -----  ------------------------------------  --------  ----------
2026-09-06T2100-T0028  automated decision-making / adminis…  0.664  SI-3 AI and Automated Decision Maki…     0.453  —
2026-09-06T2100-T0053  enforcement intellectual / intellec…  0.569  4.1 Innovation and adaptiveness in …     0.382  —
2026-09-06T2100-T0054  patent retrieval / retrieval system…  0.536  4.1 Innovation and adaptiveness in …     0.430  —
2026-09-06T2100-T0006  geographical indication / protectio…  0.689  SI-1 Geographical indications regis…     0.355  —
2026-09-06T2100-T0067  dashboard / digital project / queen…  0.524  4.2 Innovative digital and data-dri…     0.406  —
2026-09-06T2100-T0079  ai / teacher / education / learning   0.551  SI-3 AI and Automated Decision Maki…     0.391  —
2026-09-06T2100-T0031  adm / ai / deci

## Stage 4 — Opportunity index

**This is a relative, unitless, within-run ordering. It is not a market size,
it is not a dollar figure, and it cannot be converted into one.** This is the
most important caveat in the method and the easiest one to lose in a slide.

Components are percentile-ranked within the run before being combined, which is
the only way measures on this many different scales can be added at all. Two
guards are worth checking in the output below:

- Topics thinner than `opportunity_index.min_documents` are **suppressed, not
  scored**. A composite built on eight documents looks identical to one built
  on eight hundred, and that is exactly how a horizon scan misleads people.
- A component with no data has its **weight redistributed** across the
  components that do. Without that, disabling a source would silently shrink
  every index and leave the ranking looking unchanged while measuring something
  different.

The index is deliberately excluded from the ranking formula in Stage 5. It is
the weakest-founded number in the pipeline, and folding it into the headline
ordering would launder that weakness.

In [12]:
scored = [r for r in ranked_topics if not r["index_suppressed"]]
suppressed = [r for r in ranked_topics if r["index_suppressed"]]

if scored:
    weights = scored[0]["index_components"].get("_effective_weights", {})
    print("Effective component weights after redistribution:")
    print("  " + ", ".join(f"{k} {v}" for k, v in sorted(weights.items())))
    omitted = scored[0]["index_components"].get("_omitted_components") or []
    print("  omitted for lack of data: " + (", ".join(omitted) if omitted else "none"))
    print()

    keys = sorted(k for k in scored[0]["index_components"] if not k.startswith("_"))
    print(table(
        [
            (r["topic_id"], (r["label"] or "")[:36],
             *[r["index_components"].get(k) for k in keys], r["opportunity_index"])
            for r in scored
        ],
        ["topic", "label", *keys, "index"],
        max_width=36,
    ))

if suppressed:
    print()
    print(f"{len(suppressed)} topic(s) suppressed as too thin to index honestly: "
          + ", ".join(r["topic_id"] for r in suppressed))

Effective component weights after redistribution:
  attention 0.2941, attention_tone 0.1176, policy_salience 0.2353, research_growth 0.3529
  omitted for lack of data: patent_activity

topic                  label                                 attention  attention_tone  policy_salience  research_growth  index
---------------------  ------------------------------------  ---------  --------------  ---------------  ---------------  -----
2026-09-06T2100-T0028  automated decision-making / administ      0.836           0.517            0.698            0.810  0.757
2026-09-06T2100-T0053  enforcement intellectual / intellect      0.741           0.026            0.940            0.336  0.561
2026-09-06T2100-T0054  patent retrieval / retrieval system       0.147           0.672            0.655            0.845  0.575
2026-09-06T2100-T0006  geographical indication / protection      0.750           0.845            0.578            0.784  0.733
2026-09-06T2100-T0067  dashboard / digital proj

### Re-deriving the index

Recomputed from the stored components and the effective weights actually used, which is also the only way to see the redistribution having happened rather than take it on trust.

In [13]:
from src.notebook import TOLERANCE_INDEX

# The index is a plain weighted sum of stored components, so it is checkable
# the same way everything else here is. It is the one check that cannot run to
# machine precision: Stage 4 stores its components and effective weights
# rounded to 4 dp for legibility, while the index it stored was computed from
# the unrounded values. That rounding is the entire error budget below, which
# is why the tolerance is 1e-3 and not 1e-9. A deviation materially larger than
# ~1e-4 here is not rounding and should be treated as a defect.
verify_close(
    "opportunity_index",
    [r["opportunity_index"] for r in scored],
    [
        sum(r["index_components"][k] * w
            for k, w in r["index_components"]["_effective_weights"].items())
        for r in scored
    ],
    tolerance=TOLERANCE_INDEX,
)

PASS  opportunity_index reproduced for 117 topics — largest deviation 8.19e-05 (tolerance 0.001)


> **Noted automatically from this run's own numbers.**
>
> - Component(s) **patent_activity** had no data and their weight was redistributed across the rest. The index remains internally consistent but measures fewer things than the config describes — say so when presenting it, rather than quoting the configured weights.

## Stage 5 — Ranking and synthesis

The shortlist is a weighted combination of three axes — emergence, strategic
fit and asset leverage — each percentile-ranked within the run before
weighting, so that the configured weights describe what the code actually does.

The opportunity index is deliberately **not** one of the axes. It is the
weakest-founded number in the pipeline, and folding it into the headline
ordering would launder that weakness into the thing everyone reads first.

### Re-deriving the published order

The cell below reproduces the ranking by calling `stage5_synthesis.composite_scores` — the same function the pipeline called, not a reimplementation of it, so the check cannot drift away from the code it is meant to be checking. If it agrees, then given this corpus and these weights the ordering is not a matter of opinion. Whether they are the right weights is a separate question, and an open one.

In [14]:
from src.stage5_synthesis import composite_scores

# Reproduced by calling the pipeline's own ranking function, not by
# reimplementing it here. A reimplementation can drift away from the code it is
# meant to be checking; this cannot.
#
# Stage 5 read topics in emergence_score DESC order and sorted them stably, so
# the reproduction has to start from that same order to be faithful.
by_id = {r["topic_id"]: r for r in ranked_topics}
rows = [by_id[t["topic_id"]] for t in topics]

recomputed = composite_scores(rows, RANK_WEIGHTS)
verify_close("composite_rank_score", [r["composite_rank_score"] for r in rows], recomputed)

order = sorted(range(len(rows)), key=lambda i: -recomputed[i])
verify_identical(
    "shortlist ordering",
    [r["topic_id"] for r in ranked_topics],        # stored, ORDER BY rank
    [rows[i]["topic_id"] for i in order],          # recomputed
)

PASS  composite_rank_score reproduced for 117 topics — largest deviation 0 (tolerance 1e-09)
PASS  shortlist ordering reproduced — 117 of 117 entries identical


### The published shortlist

In [15]:
print(table(
    [
        (r["rank"], (r["label"] or r["topic_id"])[:42], r["horizon"], r["signal_class"],
         r["emergence_score"], r["strategic_fit"], r["asset_leverage"],
         None if r["index_suppressed"] else r["opportunity_index"],
         r["composite_rank_score"])
        for r in ranked_topics
    ],
    ["#", "topic", "H", "signal", "emrg", "fit", "lev", "index", "composite"],
    places=3, max_width=42,
))

  #  topic                                       H   signal   emrg    fit    lev  index  composite
---  ------------------------------------------  --  ------  -----  -----  -----  -----  ---------
  1  automated decision-making / administrative  H1  strong  0.689  0.664  0.453  0.757      0.925
  2  enforcement intellectual / intellectual pr  H1  strong  0.764  0.569  0.382  0.561      0.865
  3  patent retrieval / retrieval system / info  H1  latent  0.709  0.536  0.430  0.575      0.846
  4  geographical indication / protection geogr  H2  strong  0.739  0.689  0.355  0.733      0.840
  5  dashboard / digital project / queensland /  H2  strong  0.745  0.524  0.406  0.585      0.828
  6  ai / teacher / education / learning         H1  weak    0.703  0.551  0.391  0.482      0.821
  7  adm / ai / decision-making / automated      H2  strong  0.529  0.756  0.482  0.794      0.819
  8  intellectual property / corporate / proper  H2  strong  0.589  0.616  0.386  0.844      0.807
  9  artif

### Strategic fit × asset leverage

The view that separates an interesting trend from a viable venture, split at the median of each axis. Note that a median split guarantees a populated *act* quadrant whether or not anything in the run deserves one — the quadrant is a relative position, not a verdict.

In [16]:
from src.stage5_synthesis import quadrant

# Recomputed here rather than read back: Stage 5 records the quadrant in
# topics.csv but does not persist it to `topic_scores`, so the database cannot
# be asked for it. Recomputing from the stored axes and the same median split
# Stage 5 used gives the identical placement, and shows the derivation.
fit_cut = float(np.median([float(r["strategic_fit"] or 0.0) for r in ranked_topics]))
lev_cut = float(np.median([float(r["asset_leverage"] or 0.0) for r in ranked_topics]))
LABELS = ("watch", "on-strategy, no right-to-play",
          "capability looking for a problem", "act")

placement = {
    r["topic_id"]: quadrant(
        float(r["strategic_fit"] or 0.0), float(r["asset_leverage"] or 0.0),
        fit_cut, lev_cut, LABELS,
    )
    for r in ranked_topics
}

print(f"Split at the median of each axis (fit {fit_cut:.3f}, leverage {lev_cut:.3f}).\n")
for name in ("act", "on-strategy, no right-to-play",
             "capability looking for a problem", "watch"):
    members = [r for r in ranked_topics if placement[r["topic_id"]] == name]
    print(f"{name.upper()} ({len(members)})")
    for r in members:
        print(f"    {r['rank']:>3}  {(r['label'] or r['topic_id'])[:60]}")
    if not members:
        print("    (none)")
    print()

Split at the median of each axis (fit 0.473, leverage 0.351).

ACT (51)
      1  automated decision-making / administrative / government / la
      2  enforcement intellectual / intellectual property / property 
      3  patent retrieval / retrieval system / information retrieval 
      4  geographical indication / protection geographical / trade / 
      5  dashboard / digital project / queensland / contribution publ
      6  ai / teacher / education / learning
      7  adm / ai / decision-making / automated
      8  intellectual property / corporate / property protection / va
      9  artificial intelligence / patent / inventorship / intelligen
     10  patent / retrieval / document / classification
     11  intellectual property / right / property law / ip
     12  ip / law / intellectual property / chapter
     13  criminal / ip / enforcement / infringement
     14  cultural heritage / property cultural / intellectual / intan
     15  indigenous / intellectual / property / traditio

## Evidence — the documents behind the top 8 topics

Reading the primary documents is the cheapest and most reliable quality control
in this method, and the only one that finds clustering artefacts. Some topics
below will not be themes at all; they will be a set of documents that happen to
share vocabulary. Finding those is the point of this section, not a sign that
something has gone wrong.

For each topic: its scores, its defining terms, its trajectory, and the
documents nearest its centre.

In [17]:
TOPIC_ID = '2026-09-06T2100-T0028'
topic = next(r for r in ranked_topics if r["topic_id"] == TOPIC_ID)

print(f"#{topic['rank']}  {topic['label'] or TOPIC_ID}")
print(f"{topic['document_count']} documents · {topic['first_slice']}–{topic['last_slice']} · "
      f"{topic['horizon']} · {topic['signal_class']}")
print()
print("Defining terms: " + ", ".join(t for t, _ in (topic["terms"] or [])[:12]))
print()

series = db.fetch_topic_timeseries(conn, TOPIC_ID)
if series:
    peak = max(int(p["doc_count"]) for p in series) or 1
    print("Trajectory (← burst marks slices the Kleinberg automaton flagged):")
    for point in series:
        count = int(point["doc_count"])
        bar = "█" * int(round(24 * count / peak))
        print(f"  {point['time_slice']:>7} {count:5d}  {bar}"
              + ("  ← burst" if point["in_burst"] else ""))
    print()

print("Nearest documents — the primary text every score above derives from.")
print("If these are not a coherent theme, the topic is a clustering artefact")
print("and belongs in the discard pile, whatever it scored:\n")
for doc in db.fetch_topic_documents(conn, TOPIC_ID, limit=5):
    meta = " · ".join(str(x) for x in [doc["source"], doc["year"] or "", doc["venue"] or ""] if x)
    print(f"  • {(doc['title'] or '(untitled)')[:96]}")
    print(f"    {meta[:96]}")
    if doc["url"]:
        print(f"    {doc['url']}")

#1  automated decision-making / administrative / government / law
162 documents · 2018–2026 · H1 · strong

Defining terms: automated decision-making, decision-making, automated, administrative, administrative decision-making, administrative law, automated decision, government, governmental automated, decision, law, decision making

Trajectory (← burst marks slices the Kleinberg automaton flagged):
     2018     3  ████
     2019     4  █████
     2020     7  █████████
     2021     6  ████████
     2022     4  █████
     2023    18  ████████████████████████  ← burst
     2024     5  ███████
     2025     7  █████████
     2026     6  ████████

Nearest documents — the primary text every score above derives from.
If these are not a coherent theme, the topic is a clustering artefact
and belongs in the discard pile, whatever it scored:

  • Situating the Rule of Law in the Context of Automated Decision-Making
    crossref · 2023 · The Rule of Law and Automated Decision-Making
    https://d

In [18]:
TOPIC_ID = '2026-09-06T2100-T0053'
topic = next(r for r in ranked_topics if r["topic_id"] == TOPIC_ID)

print(f"#{topic['rank']}  {topic['label'] or TOPIC_ID}")
print(f"{topic['document_count']} documents · {topic['first_slice']}–{topic['last_slice']} · "
      f"{topic['horizon']} · {topic['signal_class']}")
print()
print("Defining terms: " + ", ".join(t for t, _ in (topic["terms"] or [])[:12]))
print()

series = db.fetch_topic_timeseries(conn, TOPIC_ID)
if series:
    peak = max(int(p["doc_count"]) for p in series) or 1
    print("Trajectory (← burst marks slices the Kleinberg automaton flagged):")
    for point in series:
        count = int(point["doc_count"])
        bar = "█" * int(round(24 * count / peak))
        print(f"  {point['time_slice']:>7} {count:5d}  {bar}"
              + ("  ← burst" if point["in_burst"] else ""))
    print()

print("Nearest documents — the primary text every score above derives from.")
print("If these are not a coherent theme, the topic is a clustering artefact")
print("and belongs in the discard pile, whatever it scored:\n")
for doc in db.fetch_topic_documents(conn, TOPIC_ID, limit=5):
    meta = " · ".join(str(x) for x in [doc["source"], doc["year"] or "", doc["venue"] or ""] if x)
    print(f"  • {(doc['title'] or '(untitled)')[:96]}")
    print(f"    {meta[:96]}")
    if doc["url"]:
        print(f"    {doc['url']}")

#2  enforcement intellectual / intellectual property / property right / defense sphere
93 documents · 2018–2026 · H1 · strong

Defining terms: enforcement intellectual, enforcement, intellectual property, property right, intellectual, right, property, defense sphere, responsibility violation, right defense, toward responsibility, improvement toward

Trajectory (← burst marks slices the Kleinberg automaton flagged):
     2018     4  ███  ← burst
     2019    29  ████████████████████████  ← burst
     2020     0  
     2021     2  ██
     2022     1  █
     2023     0  
     2024     0  
     2025     0  
     2026     2  ██

Nearest documents — the primary text every score above derives from.
If these are not a coherent theme, the topic is a clustering artefact
and belongs in the discard pile, whatever it scored:

  • Enforcement of Intellectual Property Rights in the EU Member States
    crossref · 2019 · book
    https://doi.org/10.1017/9781780687827
  • Enforcement of intellectual pr

In [19]:
TOPIC_ID = '2026-09-06T2100-T0054'
topic = next(r for r in ranked_topics if r["topic_id"] == TOPIC_ID)

print(f"#{topic['rank']}  {topic['label'] or TOPIC_ID}")
print(f"{topic['document_count']} documents · {topic['first_slice']}–{topic['last_slice']} · "
      f"{topic['horizon']} · {topic['signal_class']}")
print()
print("Defining terms: " + ", ".join(t for t, _ in (topic["terms"] or [])[:12]))
print()

series = db.fetch_topic_timeseries(conn, TOPIC_ID)
if series:
    peak = max(int(p["doc_count"]) for p in series) or 1
    print("Trajectory (← burst marks slices the Kleinberg automaton flagged):")
    for point in series:
        count = int(point["doc_count"])
        bar = "█" * int(round(24 * count / peak))
        print(f"  {point['time_slice']:>7} {count:5d}  {bar}"
              + ("  ← burst" if point["in_burst"] else ""))
    print()

print("Nearest documents — the primary text every score above derives from.")
print("If these are not a coherent theme, the topic is a clustering artefact")
print("and belongs in the discard pile, whatever it scored:\n")
for doc in db.fetch_topic_documents(conn, TOPIC_ID, limit=5):
    meta = " · ".join(str(x) for x in [doc["source"], doc["year"] or "", doc["venue"] or ""] if x)
    print(f"  • {(doc['title'] or '(untitled)')[:96]}")
    print(f"    {meta[:96]}")
    if doc["url"]:
        print(f"    {doc['url']}")

#3  patent retrieval / retrieval system / information retrieval / integration image
39 documents · 2018–2026 · H1 · latent

Defining terms: patent retrieval, retrieval, patent, retrieval system, information retrieval, integration image, learning prior, retrieval process, retrieval-based learning, image patent, mapreduce, retrieval-based

Trajectory (← burst marks slices the Kleinberg automaton flagged):
     2018     2  ██████
     2019     5  ███████████████
     2020     3  █████████
     2021     2  ██████
     2022     5  ███████████████
     2023     6  ██████████████████
     2024     4  ████████████
     2025     2  ██████
     2026     8  ████████████████████████

Nearest documents — the primary text every score above derives from.
If these are not a coherent theme, the topic is a clustering artefact
and belongs in the discard pile, whatever it scored:

  • Challenges in Patent Information Retrieval
    crossref · 2020 · The Information Retrieval Series
    https://doi.org/10.1

In [20]:
TOPIC_ID = '2026-09-06T2100-T0006'
topic = next(r for r in ranked_topics if r["topic_id"] == TOPIC_ID)

print(f"#{topic['rank']}  {topic['label'] or TOPIC_ID}")
print(f"{topic['document_count']} documents · {topic['first_slice']}–{topic['last_slice']} · "
      f"{topic['horizon']} · {topic['signal_class']}")
print()
print("Defining terms: " + ", ".join(t for t, _ in (topic["terms"] or [])[:12]))
print()

series = db.fetch_topic_timeseries(conn, TOPIC_ID)
if series:
    peak = max(int(p["doc_count"]) for p in series) or 1
    print("Trajectory (← burst marks slices the Kleinberg automaton flagged):")
    for point in series:
        count = int(point["doc_count"])
        bar = "█" * int(round(24 * count / peak))
        print(f"  {point['time_slice']:>7} {count:5d}  {bar}"
              + ("  ← burst" if point["in_burst"] else ""))
    print()

print("Nearest documents — the primary text every score above derives from.")
print("If these are not a coherent theme, the topic is a clustering artefact")
print("and belongs in the discard pile, whatever it scored:\n")
for doc in db.fetch_topic_documents(conn, TOPIC_ID, limit=5):
    meta = " · ".join(str(x) for x in [doc["source"], doc["year"] or "", doc["venue"] or ""] if x)
    print(f"  • {(doc['title'] or '(untitled)')[:96]}")
    print(f"    {meta[:96]}")
    if doc["url"]:
        print(f"    {doc['url']}")

#4  geographical indication / protection geographical / trade / eu
156 documents · 2018–2026 · H2 · strong

Defining terms: geographical indication, indication, geographical, protection geographical, trade, protection, eu, europe, indication protection, trade agreement, indication designation, protection eu

Trajectory (← burst marks slices the Kleinberg automaton flagged):
     2018     4  █████
     2019    18  ████████████████████████
     2020     7  █████████
     2021    11  ███████████████
     2022    11  ███████████████
     2023     6  ████████
     2024     9  ████████████
     2025    18  ████████████████████████  ← burst
     2026    13  █████████████████  ← burst

Nearest documents — the primary text every score above derives from.
If these are not a coherent theme, the topic is a clustering artefact
and belongs in the discard pile, whatever it scored:

  • The Protection of Geographical Indications
    crossref · 2019 · edited-book
    https://doi.org/10.4337/97817889754

In [21]:
TOPIC_ID = '2026-09-06T2100-T0067'
topic = next(r for r in ranked_topics if r["topic_id"] == TOPIC_ID)

print(f"#{topic['rank']}  {topic['label'] or TOPIC_ID}")
print(f"{topic['document_count']} documents · {topic['first_slice']}–{topic['last_slice']} · "
      f"{topic['horizon']} · {topic['signal_class']}")
print()
print("Defining terms: " + ", ".join(t for t, _ in (topic["terms"] or [])[:12]))
print()

series = db.fetch_topic_timeseries(conn, TOPIC_ID)
if series:
    peak = max(int(p["doc_count"]) for p in series) or 1
    print("Trajectory (← burst marks slices the Kleinberg automaton flagged):")
    for point in series:
        count = int(point["doc_count"])
        bar = "█" * int(round(24 * count / peak))
        print(f"  {point['time_slice']:>7} {count:5d}  {bar}"
              + ("  ← burst" if point["in_burst"] else ""))
    print()

print("Nearest documents — the primary text every score above derives from.")
print("If these are not a coherent theme, the topic is a clustering artefact")
print("and belongs in the discard pile, whatever it scored:\n")
for doc in db.fetch_topic_documents(conn, TOPIC_ID, limit=5):
    meta = " · ".join(str(x) for x in [doc["source"], doc["year"] or "", doc["venue"] or ""] if x)
    print(f"  • {(doc['title'] or '(untitled)')[:96]}")
    print(f"    {meta[:96]}")
    if doc["url"]:
        print(f"    {doc['url']}")

#5  dashboard / digital project / queensland / contribution published
36 documents · 2025–2026 · H2 · strong

Defining terms: dashboard, project dashboard, digital project, queensland, digital, queensland government, contribution published, dashboard contribution, dashboard faq, dashboard resource, dashboard website, enabled initiative

Trajectory (← burst marks slices the Kleinberg automaton flagged):
     2018     0  
     2019     0  
     2020     0  
     2021     0  
     2022     0  
     2023     0  
     2024     0  
     2025     6  ██████  ← burst
     2026    23  ████████████████████████  ← burst

Nearest documents — the primary text every score above derives from.
If these are not a coherent theme, the topic is a clustering artefact
and belongs in the discard pile, whatever it scored:

  • DE—Digital Projects Dashboard contribution
    datagovau · 2026 · Education
    https://data.gov.au/data/dataset/de-digital-projects-dashboard-contribution
  • DSDIP—Digital Projects Das

In [22]:
TOPIC_ID = '2026-09-06T2100-T0079'
topic = next(r for r in ranked_topics if r["topic_id"] == TOPIC_ID)

print(f"#{topic['rank']}  {topic['label'] or TOPIC_ID}")
print(f"{topic['document_count']} documents · {topic['first_slice']}–{topic['last_slice']} · "
      f"{topic['horizon']} · {topic['signal_class']}")
print()
print("Defining terms: " + ", ".join(t for t, _ in (topic["terms"] or [])[:12]))
print()

series = db.fetch_topic_timeseries(conn, TOPIC_ID)
if series:
    peak = max(int(p["doc_count"]) for p in series) or 1
    print("Trajectory (← burst marks slices the Kleinberg automaton flagged):")
    for point in series:
        count = int(point["doc_count"])
        bar = "█" * int(round(24 * count / peak))
        print(f"  {point['time_slice']:>7} {count:5d}  {bar}"
              + ("  ← burst" if point["in_burst"] else ""))
    print()

print("Nearest documents — the primary text every score above derives from.")
print("If these are not a coherent theme, the topic is a clustering artefact")
print("and belongs in the discard pile, whatever it scored:\n")
for doc in db.fetch_topic_documents(conn, TOPIC_ID, limit=5):
    meta = " · ".join(str(x) for x in [doc["source"], doc["year"] or "", doc["venue"] or ""] if x)
    print(f"  • {(doc['title'] or '(untitled)')[:96]}")
    print(f"    {meta[:96]}")
    if doc["url"]:
        print(f"    {doc['url']}")

#6  ai / teacher / education / learning
50 documents · 2019–2025 · H1 · weak

Defining terms: ai, teacher, education, aied, ai education, learning, ai chatbot, educational, edtech, chatbot, ai edtech, learning outcome

Trajectory (← burst marks slices the Kleinberg automaton flagged):
     2018     0  
     2019     1  ███
     2020     3  ████████
     2021     3  ████████
     2022     9  ████████████████████████  ← burst
     2023     6  ████████████████  ← burst
     2024     2  █████
     2025     1  ███
     2026     0  

Nearest documents — the primary text every score above derives from.
If these are not a coherent theme, the topic is a clustering artefact
and belongs in the discard pile, whatever it scored:

  • State of the art and practice in AI in education
    openalex · 2022 · European Journal of Education
    https://doi.org/10.1111/ejed.12533
  • Vision, challenges, roles and research issues of Artificial Intelligence in Education
    openalex · 2020 · Computers and Edu

In [23]:
TOPIC_ID = '2026-09-06T2100-T0031'
topic = next(r for r in ranked_topics if r["topic_id"] == TOPIC_ID)

print(f"#{topic['rank']}  {topic['label'] or TOPIC_ID}")
print(f"{topic['document_count']} documents · {topic['first_slice']}–{topic['last_slice']} · "
      f"{topic['horizon']} · {topic['signal_class']}")
print()
print("Defining terms: " + ", ".join(t for t, _ in (topic["terms"] or [])[:12]))
print()

series = db.fetch_topic_timeseries(conn, TOPIC_ID)
if series:
    peak = max(int(p["doc_count"]) for p in series) or 1
    print("Trajectory (← burst marks slices the Kleinberg automaton flagged):")
    for point in series:
        count = int(point["doc_count"])
        bar = "█" * int(round(24 * count / peak))
        print(f"  {point['time_slice']:>7} {count:5d}  {bar}"
              + ("  ← burst" if point["in_burst"] else ""))
    print()

print("Nearest documents — the primary text every score above derives from.")
print("If these are not a coherent theme, the topic is a clustering artefact")
print("and belongs in the discard pile, whatever it scored:\n")
for doc in db.fetch_topic_documents(conn, TOPIC_ID, limit=5):
    meta = " · ".join(str(x) for x in [doc["source"], doc["year"] or "", doc["venue"] or ""] if x)
    print(f"  • {(doc['title'] or '(untitled)')[:96]}")
    print(f"    {meta[:96]}")
    if doc["url"]:
        print(f"    {doc['url']}")

#7  adm / ai / decision-making / automated
99 documents · 2018–2026 · H2 · strong

Defining terms: adm, ai, decision-making, automated decision-making, automated, decision, accountability, human, law, algorithmic, automated decision, algorithm

Trajectory (← burst marks slices the Kleinberg automaton flagged):
     2018     2  █████
     2019     3  ███████
     2020     5  ████████████
     2021     6  ██████████████
     2022     7  █████████████████
     2023     8  ███████████████████
     2024     7  █████████████████
     2025     9  ██████████████████████  ← burst
     2026    10  ████████████████████████  ← burst

Nearest documents — the primary text every score above derives from.
If these are not a coherent theme, the topic is a clustering artefact
and belongs in the discard pile, whatever it scored:

  • Administrative law and the machines of government: judicial review of automated public-sector de
    crossref · 2019 · Legal Studies
    https://doi.org/10.1017/lst.2019.9
 

In [24]:
TOPIC_ID = '2026-09-06T2100-T0035'
topic = next(r for r in ranked_topics if r["topic_id"] == TOPIC_ID)

print(f"#{topic['rank']}  {topic['label'] or TOPIC_ID}")
print(f"{topic['document_count']} documents · {topic['first_slice']}–{topic['last_slice']} · "
      f"{topic['horizon']} · {topic['signal_class']}")
print()
print("Defining terms: " + ", ".join(t for t, _ in (topic["terms"] or [])[:12]))
print()

series = db.fetch_topic_timeseries(conn, TOPIC_ID)
if series:
    peak = max(int(p["doc_count"]) for p in series) or 1
    print("Trajectory (← burst marks slices the Kleinberg automaton flagged):")
    for point in series:
        count = int(point["doc_count"])
        bar = "█" * int(round(24 * count / peak))
        print(f"  {point['time_slice']:>7} {count:5d}  {bar}"
              + ("  ← burst" if point["in_burst"] else ""))
    print()

print("Nearest documents — the primary text every score above derives from.")
print("If these are not a coherent theme, the topic is a clustering artefact")
print("and belongs in the discard pile, whatever it scored:\n")
for doc in db.fetch_topic_documents(conn, TOPIC_ID, limit=5):
    meta = " · ".join(str(x) for x in [doc["source"], doc["year"] or "", doc["venue"] or ""] if x)
    print(f"  • {(doc['title'] or '(untitled)')[:96]}")
    print(f"    {meta[:96]}")
    if doc["url"]:
        print(f"    {doc['url']}")

#8  intellectual property / corporate / property protection / valuation
186 documents · 2018–2026 · H2 · strong

Defining terms: intellectual property, intellectual, property, corporate, property protection, valuation, protection, firm, asset, valuation intellectual, digital, corporate governance

Trajectory (← burst marks slices the Kleinberg automaton flagged):
     2018    10  █████████████████
     2019     3  █████
     2020     2  ███
     2021     5  █████████
     2022     4  ███████
     2023     4  ███████
     2024     4  ███████
     2025    14  ████████████████████████  ← burst
     2026     8  ██████████████  ← burst

Nearest documents — the primary text every score above derives from.
If these are not a coherent theme, the topic is a clustering artefact
and belongs in the discard pile, whatever it scored:

  • Firm market valuation and intellectual property assets
    crossref · 2019 · Industry and Innovation
    https://doi.org/10.1080/13662716.2019.1685374
  • Intellec

## What a reviewer should push on

The arithmetic above is checkable and has been checked. The judgement calls are
where this method can actually be wrong, and they are these:

1. **The scan frame.** `data/strategy/scan_frame.yaml` decides what could be
   found at all. Its STEEPV distribution mirrors where free structured data
   exists, not where opportunities are, so the scan will keep finding
   technology trends and keep missing social, values-based and environmental
   ones. A miss caused by the frame cannot be fixed by re-weighting — and
   trying is the standard way to overfit a method like this into uselessness.
2. **The weights.** Every weight was set by reading the literature and
   thinking, not by fitting to a known outcome. The test that would change
   this: take an opportunity IP Australia already pursued, set
   `collection.end_year` to the year before that work began, re-run, and see
   where it lands.
3. **The topics.** Read the evidence sections above and mark the ones that are
   not coherent themes. A clustering artefact scores exactly as confidently as
   a real topic.
4. **The thresholds.** The clustering threshold decides how much of the corpus
   is assigned to any topic at all, and the Three Horizons cut-points decide
   the band a topic is reported in. Both are recorded in the config snapshot
   above; neither is a law of nature.

Where to write the answers: `data/outputs/<run_id>/observations.yaml`. Anything
recorded there is inserted into this notebook as an analyst observation the
next time it is generated, so the reading travels with the numbers.

---

*Method: `docs/method.md`. Live project state, open issues and the calibration
log: `PROJECT_STATE.md`.*